<a href="https://colab.research.google.com/github/Shivamthakur7500/FireWatchNet/blob/main/08_ViT_Ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
BASE = "/content/drive/MyDrive/FireWatchNet"

BASE_FEATS = ["prior_7d_days", "prior_30d_days", "had_detection_prior_7d", "had_detection_prior_30d",
              "cum_days_before", "x_utm", "y_utm", "lat", "lon", "doy_sin", "doy_cos",
              "temperature_2m_c", "precipitation_mm", "volumetric_soil_water_layer_1", "wind_speed",
              "precip_7d_mm", "precip_30d_mm", "temp_7d_c", "soil_7d"]

D = pd.read_parquet(f"{BASE}/Data/processed/firewatchnet_dataset.parquet")
V = pd.read_csv(f"{BASE}/Data/processed/Satellite/ViT_Features/ViT_Features_By_Grid_Cell.csv")
vcols = [c for c in V.columns if c.startswith("vit_")]

pca = PCA(n_components=16, random_state=42).fit(V[vcols])
print("variance explained by 16 PCs:", round(pca.explained_variance_ratio_.sum(), 3))
PC = [f"vit_pc{i+1}" for i in range(16)]
pcs = pd.DataFrame(pca.transform(V[vcols]), columns=PC)
pcs.insert(0, "cell_id", V.cell_id.values)

D = D.merge(pcs, on="cell_id", how="left")
D["has_vit"] = D.vit_pc1.notna().astype("float32")
D[PC] = D[PC].fillna(0)
D[BASE_FEATS + PC] = D[BASE_FEATS + PC].astype("float32")

EVALS = {"E1": (D.split == "train", D.split == "val"),
         "E2": (D.split.isin(["train", "val"]), D.split == "test")}
SETS = {"baseline (19 features)": BASE_FEATS, "+ ViT (16 PCs + flag)": BASE_FEATS + PC + ["has_vit"]}

rows = []
for ev, (trm, tem) in EVALS.items():
    te = D[tem]
    for seed in [42, 1, 2]:
        tr = D[trm].sample(300_000, random_state=seed)
        for name, feats in SETS.items():
            m = RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=100,
                                       class_weight="balanced_subsample", n_jobs=-1, random_state=seed)
            m.fit(tr[feats], tr.fire_next_7d)
            p = m.predict_proba(te[feats])[:, 1]
            for scope, mask in [("all 2233 cells", np.ones(len(te), bool)),
                                ("1159 ViT-covered cells", te.has_vit.to_numpy() == 1)]:
                y = te.fire_next_7d.to_numpy()[mask]
                rows.append({"evaluation": ev, "scope": scope, "features": name, "seed": seed,
                             "base_rate": y.mean(), "PR_AUC": average_precision_score(y, p[mask]),
                             "ROC_AUC": roc_auc_score(y, p[mask])})
        print(ev, "seed", seed, "done")

R = pd.DataFrame(rows)
R.to_csv(f"{BASE}/Results/vit_ablation.csv", index=False)
pd.set_option("display.width", 200)
print(R.groupby(["evaluation", "scope", "features"]).agg(
      base_rate=("base_rate", "first"), PR_AUC=("PR_AUC", "mean"), PR_AUC_sd=("PR_AUC", "std"),
      ROC_AUC=("ROC_AUC", "mean")).round(4).to_string())

Mounted at /content/drive
variance explained by 16 PCs: 0.619
E1 seed 42 done
E1 seed 1 done
E1 seed 2 done
E2 seed 42 done
E2 seed 1 done
E2 seed 2 done
                                                          base_rate  PR_AUC  PR_AUC_sd  ROC_AUC
evaluation scope                  features                                                     
E1         1159 ViT-covered cells + ViT (16 PCs + flag)      0.1414  0.3424     0.0078   0.7453
                                  baseline (19 features)     0.1414  0.3283     0.0135   0.7225
           all 2233 cells         + ViT (16 PCs + flag)      0.0978  0.2899     0.0076   0.7950
                                  baseline (19 features)     0.0978  0.2954     0.0122   0.7884
E2         1159 ViT-covered cells + ViT (16 PCs + flag)      0.0045  0.0817     0.0031   0.9345
                                  baseline (19 features)     0.0045  0.0849     0.0022   0.9367
           all 2233 cells         + ViT (16 PCs + flag)      0.0095  0.2442   